In [1]:
import os
os.environ['HADOOP_CONF_DIR'] = '/etc/hadoop/conf'
os.environ['YARN_CONF_DIR'] = '/etc/hadoop/conf'

import findspark
findspark.init()
findspark.find()

'/opt/spark'

In [2]:
from datetime import datetime, timedelta
import sys

from pyspark import SparkContext, SparkConf
from pyspark.sql import SQLContext
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType
from pyspark.sql.types import LongType

import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
                    .master("yarn") \
                    .appName(f"EventsPartitioningJob") \
                    .getOrCreate()



SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/spark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop/share/hadoop/common/lib/slf4j-log4j12-1.7.30.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]


26/09/09 18:33:49 WARN Utils: Your hostname, fv4b6eac3raht5o13oi5 resolves to a loopback address: 127.0.1.1; using 10.130.0.4 instead (on interface eth0)
26/09/09 18:33:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/09 18:33:51 WARN Utils: spark.executor.instances less than spark.dynamicAllocation.minExecutors is invalid, ignoring its setting, please update your configs.
26/09/09 18:33:56 WARN Utils: spark.executor.instances less than spark.dynamicAllocation.minExecutors is invalid, ignoring its setting, please update your configs.
26/09/09 18:33:56 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!


In [4]:
events_dir = "/user/master/data/geo/events"
geo_path = "/user/s18314377/data/geo/geo.csv"

In [74]:
## Шаг 2

def clean_coord(col_name):
    return F.regexp_replace(F.col(col_name), ",", ".").cast(DoubleType())

def get_distance_formula(obj_1_lat, obj_1_lon, obj_2_lat, obj_2_lon):
    lat1 = F.radians(F.col(obj_1_lat))
    lon1 = F.radians(F.col(obj_1_lon))
    lat2 = F.radians(clean_coord(obj_2_lat))
    lon2 = F.radians(clean_coord(obj_2_lon))
    
    r = 6371

    distance_formula = (
        F.lit(2 * r) * F.asin(
            F.sqrt(    
                F.pow(F.sin((lat2 - lat1) / 2), 2) + 
                F.cos(lat1) * F.cos(lat2) * F.pow(F.sin((lon2 - lon1) / 2), 2)
            )
        )
    )
    
    return distance_formula

def get_messages_closest_cities(df_messages, df_cities):
    df_cities_prepared = df_cities.select(
        F.col("city"),
        F.col("lat").alias("city_lat"),
        F.col("lng").alias("city_lon")
    )

    df_messages_prepared = df_messages.select(
        "event",
        "event_type",
        "date",
        F.col("lat").alias("event_lat"),
        F.col("lon").alias("event_lon")
    ).withColumn("row_unique_id", F.monotonically_increasing_id())
    

    df_joined = df_messages_prepared.crossJoin(F.broadcast(df_cities_prepared))
    
    distance_formula = get_distance_formula("event", "city")

    df_joined = df_joined.withColumn("distance", distance_formula)
    
    window = Window().partitionBy('row_unique_id').orderBy("distance")

    df_messages_closest_cities = df_joined \
                .withColumn("row_number", F.row_number().over(window)) \
                .filter(F.col("row_number") == 1) \
                .drop("row_number", "city_lat", "city_lon", "row_unique_id")
    return df_messages_closest_cities
    
    
def get_users_act_cities(df_messages_closest_cities):
    df_act_city = df_messages_closest_cities \
        .withColumn(
            "rank_act_city", 
            F.row_number().over(
                Window().partitionBy("event.message_from").orderBy(F.col("event.message_ts").desc())
            )
        ) \
        .filter(F.col("rank_act_city") == 1) \
        
    return df_act_city.select(F.col("event.message_from").alias("user_id"), F.col("city").alias("act_city"))


def get_users_home_cities(df_messages_closest_cities, days_count=27):
    df_unique_days = df_messages_closest_cities.select(
        F.col("event.message_from").alias("user_id"),
        "city",
        "date"
    ).distinct()
    
    window_user_city = Window.partitionBy("user_id", "city").orderBy("date")
    
    # Сделали нумерацию в группах по юзерам-пользователям для каждого уникального дня
    df_indexed_days = df_unique_days.withColumn("day_index", F.row_number().over(window_user_city))
    
    # Вычитаем из дат дни, равные индексу 'day_seq'. При пропуске даты значение изменится - новая непрерывная группа
    df_continuous_sequences = df_indexed_days.withColumn("seq_start_date", F.date_sub(F.col("date"), F.col("day_index")))
    
    # Группируем по условным "датам начала последовательностей"
    df_continuous_groups = (
        df_continuous_sequences
        .groupBy("user_id", "city", "seq_start_date")
        .agg(F.count("date").alias("continuous_days"))
    )

    # Фильтруем по нужному days_count 
    df_filtered = df_continuous_groups.filter(F.col("continuous_days") >= days_count)
    
    window_latest = Window.partitionBy("user_id").orderBy(F.col("seq_start_date").desc())

    df_result = (
        df_filtered
        .withColumn("island_rank", F.row_number().over(window_latest))
        .filter(F.col("island_rank") == 1)
    )
    
    return df_result.select("user_id", F.col("city").alias("home_city"))



def get_users_travel_info(df_messages_closest_cities):
    # Шаг 1. Оставляем только уникальные дни (user + date), чтобы избавиться от дублей сообщений в один день
    df_user_days = df_messages_closest_cities.select(
        F.col("event.message_from").alias("user_id"), 
        "date", 
        "city"
    ).distinct()

    # Шаг 2. Окно для отслеживания хронологии пользователя
    window_time = Window.partitionBy("user_id").orderBy("date")

    # Шаг 3. Находим город из предыдущего дня
    df_with_prev = df_user_days.withColumn("prev_city", F.lag("city", 1).over(window_time))

    # Шаг 4. Ставим 1, если пользователь переехал в другой город (или это его первая запись)
    df_with_marker = df_with_prev.withColumn(
        "is_new_travel",
        F.when(F.col("prev_city").isNull() | (F.col("city") != F.col("prev_city")), 1).otherwise(0)
    )

    # Шаг 5. Создаем уникальный ID путешествия с помощью кумулятивной суммы
    # (Для каждого нового визита это число будет увеличиваться: 1, 2, 3...)
    df_with_travel_id = df_with_marker.withColumn(
        "travel_id", 
        F.sum("is_new_travel").over(window_time)
    )

    # Шаг 6. Схлопываем дни внутри одного путешествия. Нам нужна дата начала, чтобы сохранить хронологию визитов
    df_visits = (
        df_with_travel_id
        .groupBy("user_id", "travel_id", "city")
        .agg(F.min("date").alias("travel_start_date"))
    )

    # Шаг 7. Финальное окно, чтобы собрать города по порядку начала путешествий
    window_collect = Window.partitionBy("user_id").orderBy("travel_start_date")

    df_final_metrics = (
        df_visits
        # Собираем массив городов в правильном порядке
        .withColumn("travel_array", F.collect_list("city").over(window_collect))
        # Считаем общее количество посещений (размер массива)
        .withColumn("travel_count", F.size("travel_array"))
        # Берем последнюю строку для каждого юзера, где массивы собраны полностью
        .withColumn("row_num", F.row_number().over(Window.partitionBy("user_id").orderBy(F.desc("travel_start_date"))))
        .filter(F.col("row_num") == 1)
        .select("user_id", "travel_count", "travel_array")
    )

    # Смотрим результат
    return df_final_metrics


def get_timezone_column():
    city_to_tz_map = F.create_map([
        # Сидней
        F.lit("Sydney"), F.lit("Australia/Sydney"),
        F.lit("Canberra"), F.lit("Australia/Sydney"),
        F.lit("Newcastle"), F.lit("Australia/Sydney"),
        F.lit("Wollongong"), F.lit("Australia/Sydney"),
        F.lit("Maitland"), F.lit("Australia/Sydney"),
    
        # Мельбурн
        F.lit("Melbourne"), F.lit("Australia/Melbourne"),
        F.lit("Cranbourne"), F.lit("Australia/Melbourne"),
        F.lit("Geelong"), F.lit("Australia/Melbourne"),
        F.lit("Ballarat"), F.lit("Australia/Melbourne"),
        F.lit("Bendigo"), F.lit("Australia/Melbourne"),
    
        # Брисбен
        F.lit("Brisbane"), F.lit("Australia/Brisbane"),
        F.lit("Gold Coast"), F.lit("Australia/Brisbane"),
        F.lit("Townsville"), F.lit("Australia/Brisbane"),
        F.lit("Ipswich"), F.lit("Australia/Brisbane"),
        F.lit("Cairns"), F.lit("Australia/Brisbane"),
        F.lit("Toowoomba"), F.lit("Australia/Brisbane"),
        F.lit("Mackay"), F.lit("Australia/Brisbane"),
        F.lit("Rockhampton"), F.lit("Australia/Brisbane"),
    
        # Перт
        F.lit("Perth"), F.lit("Australia/Perth"),
        F.lit("Bunbury"), F.lit("Australia/Perth"),
    
        # Аделаида
        F.lit("Adelaide"), F.lit("Australia/Adelaide"),
    
        # Хобарт
        F.lit("Hobart"), F.lit("Australia/Hobart"),
        F.lit("Launceston"), F.lit("Australia/Hobart"),
    
        # Дарвин
        F.lit("Darwin"), F.lit("Australia/Darwin")
    ])
    return F.coalesce(
                city_to_tz_map[F.initcap(F.trim(F.col("city")))], 
                F.lit("Australia/Sydney") 
            )


def get_user_local_time(df_messages_closest_cities):




    df_local_time = (
        df_messages_closest_cities
        .withColumn(
            "TIME_UTC", 
            F.date_format(
                F.coalesce(
                    F.col("event.message_ts"), 
                    F.col("event.datetime")
                ).cast("timestamp"),
                "yyyy-MM-dd HH:mm:ss"
            ).cast("timestamp")
        )
        .withColumn(
            "timezone", get_timezone_column() 
        )
    )

    df_local_time_latest = (
        df_local_time
        .select(F.col("event.message_from").alias("user_id"), "TIME_UTC", "timezone")
        .withColumn("local_time", F.from_utc_timestamp(F.col("TIME_UTC"),F.col('timezone')))
        .withColumn(
            "rank", 
            F.row_number().over(
                Window().partitionBy("user_id").orderBy(F.col("TIME_UTC").desc()))
        )
        .filter(F.col("rank") == 1)
        .drop("rank", "TIME_UTC", "timezone")
    )
    
    return df_local_time_latest


In [6]:
def main(events_dir, geo_path, home_days_count=27):
    df_events = (
        spark.read
        .option("pathGlobFilter", "*part-000*.parquet") # Читаем только самый первый под-файл в каждой папке
        .parquet(events_dir)     # Заходим во все партиции
    )
    df_messages = df_events.filter(F.col("event_type") == "message")
    df_cities = spark.read.csv(geo_path, sep=";", header=True, inferSchema=True)
    
    ### Этап 1. Ищем ближайшие города к каждому сообщению
    df_messages_closest_cities = get_messages_closest_cities(df_messages, df_cities)
    
    ### Этап 2. Ищем последний город пользователя и последний домашний город пользователя
    df_act_cities = get_users_act_cities(df_messages_closest_cities)
    df_home_cities = get_users_home_cities(df_messages_closest_cities, days_count=home_days_count)
    
    ### Этап 3. Ищем статистику путешествий
    df_travel_info = get_users_travel_info(df_messages_closest_cities)
    
    
    ### Этап 4. Ищем местное время
    df_local_time = get_user_local_time(df_messages_closest_cities)
    
    
    df_result = (
        df_act_cities
        .join(df_home_cities, on="user_id", how="outer")
        .join(df_travel_info, on="user_id", how="outer")
        .join(df_local_time, on="user_id", how="outer")
    )
    return df_result
    

In [7]:
# df_res = main(events_dir, geo_path, home_days_count=3)

In [8]:
# df_res.cache()

In [ ]:
# df_res.orderBy("user_id").show(1000)

In [ ]:
df_events = (
        spark.read
        .option("pathGlobFilter", "*part-000*.parquet") # Читаем только самый первый под-файл в каждой папке
        .parquet(events_dir)     # Заходим во все партиции
    )

df_events = df_events.sample(withReplacement=False, fraction=0.1, seed=42)
# df_events = df_events.filter(F.col("date") == "2022-01-01")


df_cities = spark.read.csv(geo_path, sep=";", header=True, inferSchema=True)

df_events_closest_cities = get_messages_closest_cities(df_events, df_cities)




In [ ]:
df_zone_stats = (
    df_events_closest_cities
    .withColumn("month", F.month(F.col("date")))
    .withColumn("week", F.weekofyear(F.col("date")))
    .withColumn(
        "first_message_ts", 
        F.min(F.when(F.col("event_type") == "message", F.col("event.message_ts"))).over(Window.partitionBy("event.message_from"))
    )

)


In [ ]:
df_zone_stats.cache()

In [ ]:
df_zone_stats.printSchema()

In [ ]:
def get_grouped_stats(df_zone_stats):

    df_stats_month = (
        df_zone_stats
        .groupBy("month", F.col("city").alias("zone_id"))
        .agg(
            F.sum(F.when(F.col("event_type") == "message", 1).otherwise(0)).alias("month_message"),
            F.sum(F.when(F.col("event_type") == "reaction", 1).otherwise(0)).alias("month_reaction"),        
            F.sum(F.when(F.col("event_type") == "subscription", 1).otherwise(0)).alias("month_subscription"),
            F.sum(
                F.when(
                    (F.col("event.message_ts") == F.col("first_message_ts")), 
                    1
                ).otherwise(0)
            ).alias("month_user")
        )
    )


    df_stats_week = (
    df_zone_stats
    .groupBy("month", "week", F.col("city").alias("zone_id"))  # Месяц здесь нужен как мостик для джойна
    .agg(
        F.sum(F.when(F.col("event_type") == "message", 1).otherwise(0)).alias("week_message"),
        F.sum(F.when(F.col("event_type") == "reaction", 1).otherwise(0)).alias("week_reaction"),
        F.sum(F.when(F.col("event_type") == "subscription", 1).otherwise(0)).alias("week_subscription"),
        F.sum(
                F.when(
                    (F.col("event.message_ts") == F.col("first_message_ts")), 
                    1
                ).otherwise(0)
            ).alias("month_user")
        
        )
    )
    final_df = df_stats_week.join(
        df_stats_month, 
        on=["month", "zone_id"], 
        how="left"
    )
    return final_df

In [ ]:
df_stats = get_grouped_stats(df_zone_stats)

In [ ]:
df_stats.show(100)

In [ ]:
def main_2():
    df_events = (
        spark.read
        .option("pathGlobFilter", "*part-0000*.parquet") # Читаем только самый первый под-файл в каждой папке
        .parquet(events_dir)     # Заходим во все партиции
    )

    df_events = df_events.sample(withReplacement=False, fraction=0.1, seed=42)
    # df_events = df_events.filter(F.col("date") == "2022-01-01")


    df_cities = spark.read.csv(geo_path, sep=";", header=True, inferSchema=True)

    df_events_closest_cities = get_messages_closest_cities(df_events, df_cities)
    
    df_stats = get_grouped_stats(df_zone_stats)
    
    return df_stats
    

In [ ]:
df_stats = main_2()

In [ ]:
df_stats.cache()

## STEP 4

In [42]:
df_events = (
        spark.read
        .option("pathGlobFilter", "*part-000*.parquet") # Читаем только самый первый под-файл в каждой папке
        .parquet(events_dir)     # Заходим во все партиции
#         .filter(F.col("date").isin(["2022-05-01", "2022-05-02", "2022-05-03"]))
)

df_events = df_events.sample(withReplacement=False, fraction=0.1, seed=42)


df_cities = spark.read.csv(geo_path, sep=";", header=True, inferSchema=True)
df_events_closest_cities.unpersist()
df_events_closest_cities = get_messages_closest_cities(df_events, df_cities)

In [43]:
df_events_closest_cities.cache()

DataFrame[event: struct<admins:array<bigint>,channel_id:bigint,datetime:string,media:struct<media_type:string,src:string>,message:string,message_channel_to:bigint,message_from:bigint,message_group:bigint,message_id:bigint,message_to:bigint,message_ts:string,reaction_from:string,reaction_type:string,subscription_channel:bigint,subscription_user:string,tags:array<string>,user:string>, event_type: string, date: date, event_lat: double, event_lon: double, city: string, distance: double]

In [44]:


def get_subscriptions_users_unique_pairs(df_events):
    df_users_subscriptions = (
        df_events
        .filter(F.col("event_type") == "subscription")
        .select(
            F.col("event.user").alias("user_id"),
            F.col("event.subscription_channel")
        ).distinct()
    )
    df_user_pairs_all = df_users_subscriptions.alias("df1").join(
        df_users_subscriptions.alias("df2"),
        F.col("df1.subscription_channel") == F.col("df2.subscription_channel")
    )

    df_subscriptions_user_pairs = (
        df_user_pairs_all
        .filter(
            F.col("df1.user_id").cast(LongType()) < F.col("df2.user_id").cast(LongType())
        )
        .select(
            F.col("df1.user_id").alias("user_left"), 
            F.col("df2.user_id").alias("user_right")
        )
        .distinct()
    )
    
    return df_subscriptions_user_pairs


In [45]:
df_subscriptions_user_pairs = get_subscriptions_users_unique_pairs(df_events_closest_cities)


In [46]:
df_subscriptions_user_pairs.summary().show()

+-------+------------------+------------------+
|summary|         user_left|        user_right|
+-------+------------------+------------------+
|  count|          53891326|          53891326|
|   mean|56706.793462996255|113410.09717339299|
| stddev|40113.436067458024| 40064.34578357984|
|    min|                 0|               100|
|    25%|           22784.0|           85164.0|
|    50%|           49801.0|          120326.0|
|    75%|           85170.0|          147331.0|
|    max|             99999|             99999|
+-------+------------------+------------------+



In [47]:
def get_messages_users_unique_pairs(df_events):
    
    df_messages_user_pairs = (
        df_events
        .filter(F.col("event_type") == "message")
        .filter(F.col("event.message_to").isNotNull())
        .select(
            F.col("event.message_from").alias("user_left"), 
            F.col("event.message_to").alias("user_right")
        )
        .distinct()
    )
    
    return df_messages_user_pairs


df_messages_user_pairs = get_messages_users_unique_pairs(df_events_closest_cities)


In [48]:
df_messages_user_pairs.summary().show()

+-------+------------------+------------------+
|summary|         user_left|        user_right|
+-------+------------------+------------------+
|  count|              1039|              1039|
|   mean| 84948.00673724736|109322.88354186718|
| stddev|46289.523198901625| 46799.17538075638|
|    min|               198|               661|
|    25%|             47238|             73763|
|    50%|             87383|            131533|
|    75%|            126334|            149488|
|    max|            162999|            149488|
+-------+------------------+------------------+



In [49]:
# Шаг 1: Вычитаем переписки в направлении user_id_1 -> user_id_2
df_pairs = df_subscriptions_user_pairs.join(
    df_messages_user_pairs,
    (df_subscriptions_user_pairs["user_left"] == df_messages_user_pairs["user_left"]) & 
    (df_subscriptions_user_pairs["user_right"] == df_messages_user_pairs["user_right"]),
    how="left_anti"
)

# Шаг 2: Из результата вычитаем переписки в обратном направлении user_id_2 -> user_id_1
df_pairs = df_pairs.join(
    df_messages_user_pairs,
    (df_pairs["user_left"] == df_messages_user_pairs["user_right"]) & 
    (df_pairs["user_right"] == df_messages_user_pairs["user_left"]),
    how="left_anti"
)

In [50]:
df_pairs.summary().show()

+-------+------------------+------------------+
|summary|         user_left|        user_right|
+-------+------------------+------------------+
|  count|          53891323|          53891323|
|   mean|56706.791317555886|113410.09624818452|
| stddev|40113.435220682084| 40064.34616771169|
|    min|                 0|               100|
|    25%|           22783.0|           85163.0|
|    50%|           49802.0|          120325.0|
|    75%|           85170.0|          147332.0|
|    max|             99999|             99999|
+-------+------------------+------------------+



In [51]:
53891326 - 53891323

3

In [76]:
df_users_latest_message = (
    df_events_closest_cities
    .filter(F.col("event_type") == "message")
    .withColumn(
        "TIME_UTC", 
            F.date_format(
                F.coalesce(
                    F.col("event.message_ts"), 
                    F.col("event.datetime")
                ).cast("timestamp"),
                "yyyy-MM-dd HH:mm:ss"
            ).cast("timestamp")
    )
    .withColumn(
        "timezone", get_timezone_column()
    )
    .withColumn(
        "rank", 
        F.row_number()
        .over(
            Window().partitionBy(F.col("event.message_from")).orderBy(F.col("TIME_UTC").desc())
        )
    )
    .filter(F.col("rank") == 1)
    .withColumn("local_time", F.from_utc_timestamp(F.col("TIME_UTC"),F.col('timezone')))

#     .select(
#         F.col("event.message_from").alias("user_id"),
#         F.col("event_lat"),
#         F.col("event_lon")
#     )
)

In [77]:
df_users_latest_message.summary().show()

+-------+----------+-------------------+------------------+----------+------------------+------------------+----+
|summary|event_type|          event_lat|         event_lon|      city|          distance|          timezone|rank|
+-------+----------+-------------------+------------------+----------+------------------+------------------+----+
|  count|      1412|               1412|              1412|      1412|              1412|              1412|1412|
|   mean|      null|-30.812789121958872| 145.2467959414498|      null| 68.77837585384538|              null| 1.0|
| stddev|      null|  7.670882356240444|10.156660839456078|      null|30.956650391847592|              null| 0.0|
|    min|   message| -42.87469876778056|115.66314698658805|  Adelaide|0.4932841715540931|Australia/Adelaide|   1|
|    25%|      null| -36.85976006157394|144.72576968447797|      null| 44.69131314294546|              null|   1|
|    50%|      null|-32.765742503847626|147.68185492784295|      null| 68.19218164572726

In [82]:
from pyspark.sql import functions as F

final_pairs_df = df_pairs
# 1. Готовим компактную табличку с координатами юзеров
df_user_coords = df_users_latest_message.select(
    F.col("event.message_from").alias("user_id"),
    F.col("event_lat").alias("user_lat"),   # подставь свои названия колонок, если они другие
    F.col("event_lon").alias("user_lon"),
    F.col("city").alias("zone_id"),
    F.col("local_time")
)

# 2. Джойним координаты для ЛЕВОГО юзера (user_left)
# Используем F.broadcast(), чтобы Spark работал молниеносно
df_with_coord_left = final_pairs_df.join(
    F.broadcast(df_user_coords.alias("u_left")),
    final_pairs_df["user_left"] == F.col("u_left.user_id"),
    how="inner"
).select(
    final_pairs_df["user_left"],
    final_pairs_df["user_right"],
    F.col("u_left.user_lat").alias("lat_left"),
    F.col("u_left.user_lon").alias("lon_left"),
    "u_left.zone_id", "u_left.local_time"
)

# 3. Джойним координаты для ПРАВОГО юзера (user_right)
df_with_all_coords = df_with_coord_left.join(
    F.broadcast(df_user_coords.alias("u_right")),
    df_with_coord_left["user_right"] == F.col("u_right.user_id"),
    how="inner"
).select(
    df_with_coord_left["user_left"],
    df_with_coord_left["user_right"],
    df_with_coord_left["lat_left"],
    df_with_coord_left["lon_left"],
    F.col("u_right.user_lat").alias("lat_right"),
    F.col("u_right.user_lon").alias("lon_right"),
    "u_left.zone_id", "u_left.local_time"

)

# 4. Считаем дистанцию и фильтруем пары в радиусе 1 км
distance_formula = get_distance_formula("lat_left", "lon_left", "lat_right", "lon_right")

close_pairs_df = (
    df_with_all_coords
    .withColumn("distance", distance_formula) 
    .filter(F.col("distance") <= 100)
    .withColumn("processed_dttm", F.current_timestamp())
    .select("user_left", "user_right", "processed_dttm", "zone_id", "local_time")

)
close_pairs_df.show()


+---------+----------+--------------------+----------+-------------------+
|user_left|user_right|      processed_dttm|   zone_id|         local_time|
+---------+----------+--------------------+----------+-------------------+
|   115438|    124586|2026-09-09 19:02:...| Newcastle|2022-05-21 20:29:36|
|    52486|    113950|2026-09-09 19:02:...|   Bendigo|2022-05-18 17:37:51|
|    42261|    143998|2026-09-09 19:02:...|   Bendigo|2021-05-19 03:24:51|
|    82707|    106719|2026-09-09 19:02:...|Gold Coast|2021-05-06 01:45:21|
|    61319|     96528|2026-09-09 19:02:...|  Maitland|2022-05-25 10:57:25|
|     5129|    100045|2026-09-09 19:02:...|Townsville|2022-05-29 03:56:35|
|    49158|     51575|2026-09-09 19:02:...| Melbourne|2022-05-18 18:26:20|
|    14728|     30993|2026-09-09 19:02:...| Melbourne|2021-04-09 05:32:09|
|     8078|    114610|2026-09-09 19:02:...|   Bendigo|2022-05-24 22:13:20|
|    49528|     60573|2026-09-09 19:02:...|Launceston|2021-05-28 13:29:12|
|     8683|     39251|202

In [73]:
close_pairs_df.select("distance").summary().show()

+-------+------------------+
|summary|          distance|
+-------+------------------+
|  count|              3697|
|   mean| 1523.808391225161|
| stddev| 1074.871278841495|
|    min|3.9479587047068887|
|    25%| 657.5440896300585|
|    50%|1337.6168293892856|
|    75%| 2374.513267793719|
|    max| 3807.132425557304|
+-------+------------------+



In [53]:
df_users_latest_message.show()

+-------+-------------------+------------------+
|user_id|          event_lat|         event_lon|
+-------+-------------------+------------------+
|    120|  -32.7236875894286|116.08146269884782|
|    198|-21.125079671479728|150.06975449298832|
|    273|  -32.5329873261993|116.63108866194814|
|    331| -32.89656496998274| 151.5737614525046|
|    383|-37.480837804855845|145.24420047622348|
|    389| -27.59565248152552|153.95160800426046|
|    507| -41.92399623545017| 148.3034215864519|
|    589|  -42.1113781871939|147.54470970941858|
|    624| -37.64219165396576| 145.5550285558019|
|    625|-23.297294539715786|150.69110610828344|
|    891| -37.41027124566149|145.31442964524828|
|   1186| -37.80054491688759| 145.4298044532377|
|   1268| -37.10473517552611| 145.5715768957939|
|   1271|-26.872541682771523|152.92489770563054|
|   1384|-11.965507090163365| 131.6187652036733|
|   1693| -20.63064672918276| 150.1316251630897|
|   1837| -23.32382025806573|150.74772074897325|
|   1993| -37.152295

In [ ]:
df_pairs.summary().show()

In [ ]:
11210816

In [ ]:
df_user_messages.show()

In [ ]:
df_user_latest_message.show(10)